# Lab 03 — สร้าง Pipeline กันโปร่งใส (Titanic, Leak-free)

**Machine Learning with scikit-learn — Session 03**

## 🎯 วัตถุประสงค์
1. สร้าง `ColumnTransformer + Pipeline` ฉบับของตัวเอง
2. พิสูจน์ด้วยตัวเลขว่า scaling สำคัญกับ KNN แค่ไหน
3. วัดผล feature engineering อย่างซื่อสัตย์ (5-fold CV)
4. สัมผัส "ความรู้สึกเมื่อ score บวมจาก leakage" แล้วรู้จักตัวมัน

## 📝 คำแนะนำ
- กฎเหล็กของ lab นี้: **แบ่ง train/test ก่อน แล้วค่อยทำอย่างอื่น**
- ใช้ demo 03 เป็นแผนที่ได้ แต่พิมพ์เองจะจำได้แรงกว่า copy

## ⚙️ Setup — แบ่งข้อมูลก่อนทำอย่างอื่น

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age",
                                  "sibsp", "parch", "fare", "embarked"]].drop_duplicates()

X = df.drop(columns="survived")
y = df["survived"]

# TODO: แบ่ง train/test สัดส่วน 80:20 (stratify=y, random_state=42)


num_cols = ["age", "fare", "sibsp", "parch", "pclass"]
cat_cols = ["sex", "embarked"]

## โจทย์ 1 — สร้าง Pipeline เต็มรูปแบบ

เติมโค้ดให้ครบ: `num_pipe` (median impute + scale) · `cat_pipe` (most_frequent + one-hot)
· `pre` (ColumnTransformer) · `clf` (Pipeline ต่อ KNN) แล้ว fit + พิมพ์คะแนน test

In [ ]:
# TODO: สร้าง num_pipe, cat_pipe, pre, clf แล้ว fit กับ train และพิมพ์ test score

## โจทย์ 2 — พิสูจน์ว่า scaling สำคัญ

สร้าง pipeline แบบ**ไม่มี** StandardScaler แล้วเทียบคะแนนกับข้อ 1

**เขียนตอบ:** ต่างกันกี่คะแนน? แล้วทำไมโมเดล tree-based อย่าง RandomForest ถึงไม่สน scale?

In [ ]:
# TODO: pipeline แบบไม่ scale + เทียบผล

## โจทย์ 3 — Feature engineering แล้ววัดผล

เพิ่ม `family_size = sibsp + parch + 1` และ `is_child = age < 12` แล้ววัดด้วย 5-fold CV
ว่าตัวเลขขยับหรือเปล่า (เทียบกับชุดฟีเจอร์เดิมด้วย CV เดียวกัน)

**ช่วยเหลือ:** สร้าง X2 = X.copy() ก่อน แล้วใช้ cross_val_score(pipe, X2, y, cv=5)

In [ ]:
from sklearn.model_selection import cross_val_score

# TODO: สร้างฟีเจอร์ใหม่ แล้วเทียบ CV ของชุดเดิม vs ชุดใหม่

## โจทย์ 4 (โจทย์ชวนคิด) — ทำ leakage ด้วยมือตัวเอง

สร้างคอลัมน์ `lucky_ticket` = y ที่ถูกพลิกคำตอบสุ่ม 15% (เหมือน demo)
เติมเข้าไปในฟีเจอร์ แล้วดูคะแนนว่าบวมแค่ไหน — **จากนั้นลบทิ้งและเขียนสรุปเป็นภาษาไทย 3 บรรทัด** ว่า leakage คืออะไร และเจอได้จากไหนบ้างในงานจริง

In [ ]:
# TODO: สร้าง lucky_ticket, วัดคะแนนที่บวมขึ้น, ลบทิ้ง, เขียนสรุป

## 🤖 ลองถาม AI (vibe coding corner)

> "เขียน sklearn ColumnTransformer ที่คอลัมน์ตัวเลข (age, fare) ใช้ median impute + StandardScaler
> แต่คอลัมน์ category (sex, embarked) ใช้ most_frequent + OneHotEncoder(handle_unknown='ignore')
> แล้วต่อกับ KNeighborsClassifier ใน Pipeline เดียว"

รับโค้ดมาแล้ว**อ่านทุกบรรทัด** — เช็กว่า: imputer/scaler fit ที่ train เท่านั้นไหม? one-hot เผื่อ category ใหม่ไหม?
นี่คือจุดที่ AI มักพลาดเงียบ ๆ ที่สุด